In [ ]:
"""Cancelling transactions

Demonstration of how to use the CancelTransactions endpoint to cancel transactions in a LUSID portfolio.

Attributes
----------
transactions
cocoon - seed_data
holdings
cancel transactions
"""

# Cancelling transactions in LUSID

This notebooks shows how you can use the [<b>cancel transactions</b>](https://www.lusid.com/docs/api/#operation/CancelTransactions) endpoint to cancel transactions in a LUSID portfolio. For the pruposes of this demo, we will first seed a portfolio with 21 transactions spanning cash and various FTSE 100 stocks. We will then cancel 1 <i> Aviva</i> transaction and 2 <i> BHP </i> transactions.

### Setup LUSID

In [ ]:
# Import general purpose packages
import os
import json
from datetime import datetime, timedelta
import pytz

# Import lusid specific packages
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame
from finbourne_sdk_utils.cocoon.seed_sample_data import seed_data
from finbourne_sdk_utils.cocoon.utilities import create_scope_id

# Import data wrangling packages
import pandas as pd

pd.set_option("display.max_columns", None)

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

# Initiate an API Factory which is the client side object for interacting with LUSID APIs
api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook",
)

In [ ]:
# Load a mapping file for DataFrame headers for the build transaction response

with open(r"config/build_transactions_mapping.json") as mappings_file:
    build_transactions_json_mapping = json.load(mappings_file)

# Load a mapping file for DataFrame headers for the get holdings response

with open(r"config/get_holdings_mapping.json") as mappings_file:
    get_holdings_json_mapping = json.load(mappings_file)

### 1) Load default transactions into a new scope

In [ ]:
# Create a new scope

scope = create_scope_id()
portfolio_code = "EQUITY_UK"

In [ ]:
# Load a file of equity transactions

transactions_file = r"data/cancels/equity_transactions.csv"
transactions_df = pd.read_csv(transactions_file)
transactions_df["portfolio_code"] = portfolio_code

In [ ]:
# Load portfolios, instruments, and transactions

seed_data_response = seed_data(
    api_factory,
    ["portfolios", "instruments", "transactions"],
    scope,
    transactions_file,
    "csv",
)

In [ ]:
# Define the transaction portfolio API

txn_port_api = api_factory.build(lu.TransactionPortfoliosApi)

### 2) Lets check our holdings

We have:

* 132,000 units in Aviva
* 120,000 units in BHP

In [ ]:
response = txn_port_api.get_holdings(
    scope=scope, code=portfolio_code, property_keys=["Instrument/default/Name"]
)

holdings_df = lusid_response_to_data_frame(
    response, rename_properties=True, column_name_mapping=get_holdings_json_mapping
)

holdings_df

### 3) What transactions make up our Aviva and BHP holdings?

In [ ]:
build_transactions_response = txn_port_api.build_transactions(
    scope=scope,
    code=portfolio_code,
    transaction_query_parameters=models.TransactionQueryParameters(
        start_date="2020-01-01", end_date="2020-12-31"
    ),
    property_keys=["Instrument/default/Name"],
)


build_transactions_df = lusid_response_to_data_frame(
    build_transactions_response,
    rename_properties=True,
    column_name_mapping=build_transactions_json_mapping,
)
build_transactions_df.head(5)

### 4) Cancel one of the Aviva transactions

Cancel an Aviva transaction of 120,000 units

In [ ]:
cancel_response = txn_port_api.cancel_transactions(
    scope=scope, code=portfolio_code, transaction_ids=["trd_0001"]
)

first_cancel_datetime = cancel_response.as_at

print(f"The first cancel datetime request is: {first_cancel_datetime}")

### 5) Cancel two of the BHP transactions

Cancel two BHP transactions, both for 60,000 units.

In [ ]:
cancel_response = txn_port_api.cancel_transactions(
    scope=scope, code=portfolio_code, transaction_ids=["trd_0003", "trd_0004"]
)

second_cancel_datetime = cancel_response.as_at

print(f"The second cancel datetime request is: {first_cancel_datetime}")

### 6) Check holdings again

We can see two updates:

* Holdings in Aviva have been reduced 
* We no longer have any BHP holdings

In [ ]:
response = txn_port_api.get_holdings(
    scope=scope, code=portfolio_code, property_keys=["Instrument/default/Name"]
)


holdings_df = lusid_response_to_data_frame(
    response, rename_properties=True, column_name_mapping=get_holdings_json_mapping
)

holdings_df

### 7) Check holdings with AsAt just before the cancellation

The portfolio still owns:

* 132,000 units Aviva
* 120,000 units BHP

In [ ]:
as_at_time = first_cancel_datetime - timedelta(milliseconds=10)

print(f"The current datetime is {datetime.now(tz=pytz.UTC)}...")
print(f"Getting holdings as at {as_at_time}...")
print(
    f"This is just before the first cancellation datetime of {first_cancel_datetime}..."
)


response = txn_port_api.get_holdings(
    scope=scope,
    code=portfolio_code,
    property_keys=["Instrument/default/Name"],
    as_at=as_at_time,
)

holdings_df = lusid_response_to_data_frame(
    response, rename_properties=True, column_name_mapping=get_holdings_json_mapping
)

holdings_df